# Calphy Executor Integration Demo -- composition_scaling

This notebook demonstrates `mode="composition_scaling"` in `phase_diagram_workflows`: an
alchemical free-energy calculation that transforms part of a structure from one composition to
another, using Monte Carlo identity-exchange swap moves between MD blocks
(`monte_carlo.use_custom_lammps=True`, requires the `thermoatoms/lammps` fork). Both sections
below use `execution_mode="library"` -- LAMMPS is driven through a live `pylammpsmpi.LammpsLibrary`
session rather than calphy's default executable-driving mode -- and differ only in who owns that
session: calphy itself (traditional usage) or an externally-managed `executorlib` executor
(executor-based usage).</cell id="cs-demo-header">


In [1]:
import sys
from pathlib import Path

# Add project root to path for imports
PROJECT_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "pyproject.toml").is_file()),
    Path.cwd(),
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from phase_diagram_workflows.free_energies import ti_calculator as calphy_calc
from executorlib import SingleNodeExecutor
from pylammpsmpi import LammpsLibrary, init_function

EXAMPLE_ROOT = PROJECT_ROOT / "notebooks" / "Aluminium_composition_scaling"

In [2]:
from ase.build import bulk
from lammpsparser import get_potential_by_name
import pandas as pd

## 1. Structure and Potential Setup

In [3]:
# Create Aluminum structure
Al_pure_structure = bulk('Al', cubic=True).repeat(4)
N_ATOMS = len(Al_pure_structure)
print(f"Created Al structure with {N_ATOMS} atoms")

Created Al structure with 256 atoms


In [4]:
# Get potential for Al-Mg
potential_df = get_potential_by_name('2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1')
potential_df = potential_df.to_frame().transpose()
print("Potential loaded successfully:")
display(potential_df.head())

Potential loaded successfully:


/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/lammpsparser/potential.py:326: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pot["Config"] = config_lst


,Config,Filename,Model,Name,Species,Citations
164,"[pair_style eam/fs, pair_coeff * * /cmmc/ptmp/...",[potential_LAMMPS/2009--Mendelev-M-I--Al-Mg--L...,NISTiprpy,2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1,"[Al, Mg]",[{'Mendelev_2009': {'title': 'Development of i...


## 2. Traditional Usage (execution_mode="library", no external executor)

Alchemically converts 5 of the 256 Al atoms to Mg (x_Mg = 5/256 ≈ 0.0195), starting from a
reference composition of pure Al (x_Mg=0). No `lmp=` is passed below, so calphy builds and owns
its own internal `pylammpsmpi.LammpsLibrary` session. The step counts are intentionally modest,
chosen to keep this demo fast rather than to give a converged production free energy -- see Menon
et al. 2026 Sec 2.2 for the method's converged recipe (250,000 total swap attempts, 25 ps
switching time).

In [5]:
N_MG_TARGET = 5

input_params_cs = {
    "mode": "composition_scaling",
    "temperature": 300,
    "n_equilibration_steps": 5000,
    "n_switching_steps": 5000,
    "n_print_steps": 500,
    "equilibration_control": "nose-hoover",
    "execution_mode": "library",
    "queue": {
        "cores": 1,
        "scheduler": "local"
    },
    'reference_phase': 'solid',
    'file_format': 'lammps-data',
    'reference_composition': 0.0,
    'composition_scaling': {
        'output_chemical_composition': {'Al': N_ATOMS - N_MG_TARGET, 'Mg': N_MG_TARGET},
    },
    'monte_carlo': {
        'n_steps': 20,
        'n_swaps': 20,
        'use_custom_lammps': True,
    },
}

In [6]:
# Traditional usage - no executor, no metadata
print("Running traditional composition_scaling calphy calculation...")
calphy_results_traditional = calphy_calc.calc_free_energy_with_calphy(
    input_structure=Al_pure_structure,
    potential_df=potential_df,
    calphy_parameters=input_params_cs,
    working_directory=str(EXAMPLE_ROOT / "Al_traditional"),
)
print("Traditional calculation completed successfully!")

Running traditional composition_scaling calphy calculation...
Input validation successful. Proceeding with calculation.


/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(
/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Unexpected Value)
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [field_name='pressure', input_value=0, input_type=int])
  PydanticSerializationUnexpectedValue(Expected `list[list[float]]` - serialized value may not be as expected [field_name='pressure', input_value=0, input_type=int])
  return self.__pydantic_serializer__.to_python(


2026-09-04 11:35:24.010448: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-09-04 11:35:42.078873: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Traditional calculation completed successfully!


In [7]:
# Show traditional results
print("Calculation object:")
display(calphy_results_traditional[0])
print("\nResults DataFrame:")
display(calphy_results_traditional[1])

Calculation object:


Calculation(monte_carlo=MonteCarlo(n_steps=20, n_swaps=20, forward_swap_types=[], reverse_swap_types=[], allow_all_swaps=True, use_custom_lammps=True), composition_scaling=CompositionScaling(output_chemical_composition={'Al': 251, 'Mg': 5}, restrictions=[]), md=MD(timestep=0.001, n_small_steps=10000, n_every_steps=10, n_repeat_steps=10, n_cycles=100, thermostat_damping=0.1, barostat_damping=0.1, cmdargs='', init_commands=[], seed=None), nose_hoover=NoseHoover(thermostat_damping=0.1, barostat_damping=0.1), berendsen=Berendsen(thermostat_damping=100.0, barostat_damping=100.0), quantum_thermal_bath=QuantumThermalBath(thermostat_damping=0.1, barostat_damping=0.1, f_max=200.0, n_f=100), queue=Queue(scheduler='local', cores=1, jobname='calphy', walltime='23:59:00', queuename='', memory='3GB', commands=[], options={}), tolerance=Tolerance(lattice_constant=0.0002, spring_constant=0.1, solid_fraction=0.0, liquid_fraction=1.0, dissipation=0.001, pressure=10.0), phase_transition_detection=PhaseTr


Results DataFrame:


,calculation,calculation_mode,reference_phase,pressure,status,composition,temperature,free_energy,free_energy_error,forward_energy_diff,backward_energy_diff,forward_lambda,backward_lambda
0,Al_traditional,composition_scaling,solid,0,True,"{'Al': 0.98046875, 'Mg': 0.01953125}",300.0,0.041368,0.0,None,None,None,None


## 3. Executor-Based Usage (execution_mode="library", cores=2, externally-managed session)

Same calculation and same `execution_mode="library"` as above, but this time the
`pylammpsmpi.LammpsLibrary` session is built on an `executorlib.SingleNodeExecutor` you control,
instead of being owned by calphy internally. This is the pattern to use when LAMMPS needs to run
under an executor you manage -- e.g. to share one resource allocation across multiple calls, or as
the basis for scaling out to a cluster executor (`SlurmClusterExecutor`) instead of running
locally.

In [8]:
# Set up executor and LAMMPS library
print("Setting up executor...")
EXECUTOR_CORES = 2
executor = SingleNodeExecutor(
    block_allocation=True,
    hostname_localhost=True,
    max_workers=1,
    init_function=init_function,
    cache_directory=str(EXAMPLE_ROOT / "executorlib_cache"),
    resource_dict={
        "cores": EXECUTOR_CORES,
        "cwd": str(EXAMPLE_ROOT / "Al_executor"),
    },
)

# Create LAMMPS library with executor
lmp = LammpsLibrary(cores=EXECUTOR_CORES, executor=executor)
print("Executor and LAMMPS library ready!")

Setting up executor...


LAMMPS (11 Feb 2026 - MCnoforce-localE)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread.
  using 1 OpenMP thread(s) per MPI task
Executor and LAMMPS library ready!


2026-09-04 11:36:29.367307: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-04 11:36:29.490820: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [9]:
# New usage with executor and metadata
print("Running composition_scaling calphy calculation with executor...")
metadata_dict = {
    'project': 'calphy-composition-scaling-demo',
    'material': 'Al-Mg',
    'temperature': 300,
    'method': 'executor-based',
    'version': '1.0'
}

# lmp= is only honored when execution_mode="library" (already set in input_params_cs above);
# otherwise calphy ignores it and drives LAMMPS with its own default runner instead.
# queue.cores is bumped to match EXECUTOR_CORES.
input_params_cs_executor = {
    **input_params_cs,
    "queue": {**input_params_cs["queue"], "cores": EXECUTOR_CORES},
}

calphy_results_executor = calphy_calc.calc_free_energy_with_calphy(
    input_structure=Al_pure_structure,
    potential_df=potential_df,
    calphy_parameters=input_params_cs_executor,
    working_directory=str(EXAMPLE_ROOT / "Al_executor"),
    lmp=lmp,  # Pass the LAMMPS library with executor
    metadata_dict=metadata_dict  # Pass metadata for caching
)
print("Executor-based calculation completed successfully!")

Running composition_scaling calphy calculation with executor...
Input validation successful. Proceeding with calculation.


/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/ase/io/lammpsdata.py:72: FutureWarning: "style" is deprecated; please use "atom_style".
  warnings.warn(
/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Unexpected Value)
  PydanticSerializationUnexpectedValue(Expected `list[float]` - serialized value may not be as expected [field_name='pressure', input_value=0, input_type=int])
  PydanticSerializationUnexpectedValue(Expected `list[list[float]]` - serialized value may not be as expected [field_name='pressure', input_value=0, input_type=int])
  return self.__pydantic_serializer__.to_python(


OMP_NUM_THREADS environment is not set. Defaulting to 1 thread.
  using 1 OpenMP thread(s) per MPI task
Reading data file ...
  orthogonal box = (0 0 0) to (16.2 16.2 16.2)
  1 by 1 by 2 MPI processor grid
  reading atoms ...
  256 atoms
  read_data CPU = 0.004 seconds


Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 2000, page size: 100000
  master list distance cutoff = 9.5
  ghost atom cutoff = 9.5
  binsize = 4.75, bins = 4 4 4
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair eam/fs, perpetual
      attributes: half, newton on
      pair build: half/bin/atomonly/newton
      stencil: half/bin/3d
      bin: standard
Setting up Verlet run ...
  Unit style    : metal
  Current step  : 0
  Time step     : 0.001
Per MPI rank memory allocation (min/avg/max) = 3.395 | 3.395 | 3.395 Mbytes
   Step         PotEng         Press          Volume         TotEng          Temp            Lx             Ly             Lz      
         0  -873.11565     -204.07468      4251.528      -863.22725      300            16.2           16.2           16.2         
        10  -871.81783      857.60882      4254.7912     -863.22671      260.64211      16.204144      16.204144      16.204144    
        

       330  -864.3281      -1830.8995      4325.11       -857.1025       219.21452      16.292925      16.292925      16.292925    
       340  -864.53463      2065.2744      4303.6046     -856.84912      233.16755      16.265876      16.265876      16.265876    
       350  -864.97233     -2467.3794      4322.6478     -856.57278      254.8304       16.289832      16.289832      16.289832    
       360  -865.36921      2924.4803      4292.3228     -856.29604      275.26715      16.25165       16.25165       16.25165     
       370  -865.2709      -3105.5603      4323.2262     -856.02117      280.62369      16.290559      16.290559      16.290559    
       380  -864.93368      3457.9867      4292.8355     -855.76063      278.29722      16.252297      16.252297      16.252297    
       390  -864.58274     -3740.0288      4332.8677     -855.51962      274.96219      16.30266       16.30266       16.30266     
       400  -864.58409      4019.0064      4295.7028     -855.27914      282

...

      9230  -863.63584     -918.85873      4327.2513     -853.68387      301.92862      16.295613      16.295613      16.295613    
      9240  -863.59596      1097.4389      4317.4465     -853.59261      303.48723      16.283296      16.283296      16.283296    
      9250  -863.62865     -1325.751       4330.0703     -853.51196      306.92581      16.299151      16.299151      16.299151    
      9260  -863.61856      1515.8692      4316.2659     -853.44111      308.76934      16.281812      16.281812      16.281812    
      9270  -863.34285     -1699.0596      4333.6591     -853.38192      302.20052      16.303653      16.303653      16.303653    
      9280  -863.00413      1888.3684      4317.0984     -853.32706      293.58875      16.282859      16.282859      16.282859    
      9290  -862.70238     -2075.8588      4339.2315     -853.27475      286.02108      16.310638      16.310638      16.310638    
      9300  -862.69353      2163.7475      4318.5544     -853.21529      287

      9560  -863.19697     -2492.1276      4339.5885     -853.20822      303.0443       16.311085      16.311085      16.311085    
      9570  -863.39994      3223.1821      4309.2675     -853.32184      305.7551       16.273007      16.273007      16.273007    
      9580  -863.44375     -3913.1351      4344.6597     -853.44096      303.47056      16.317436      16.317436      16.317436    
      9590  -863.53558      4405.4321      4301.2142     -853.57747      302.11503      16.262864      16.262864      16.262864    
      9600  -863.45928     -4761.846       4347.8838     -853.71439      295.64607      16.321471      16.321471      16.321471    
      9610  -863.53388      5044.2583      4297.8952     -853.85045      293.78132      16.25868       16.25868       16.25868     
      9620  -863.50795     -5192.4221      4349.1473     -853.97655      289.16907      16.323052      16.323052      16.323052    
      9630  -863.61236      5262.0874      4295.159      -854.095        288

      9890  -863.34554     -525.36361      4326.0848     -853.56437      296.74668      16.294149      16.294149      16.294149    
      9900  -862.9377       755.85123      4321.7348     -853.53382      285.30035      16.288685      16.288685      16.288685    
      9910  -862.80314     -1186.608       4332.3435     -853.50286      282.1571       16.302003      16.302003      16.302003    
      9920  -863.00151      1442.2657      4317.8106     -853.46665      289.27421      16.283754      16.283754      16.283754    
      9930  -863.31653     -1786.2214      4332.6173     -853.42425      300.11785      16.302346      16.302346      16.302346    
      9940  -863.62975      2151.5098      4311.3204     -853.38886      310.69429      16.275591      16.275591      16.275591    
      9950  -863.68843     -2459.3401      4334.9599     -853.36111      313.31596      16.305284      16.305284      16.305284    
      9960  -863.57013      2836.6364      4309.6355     -853.34238      310

/cmmc/ptmp/pchilaka/Packages/calphy/calphy/runner.py:146: UserWarning: loadtxt: input contained no data: "/cmmc/u/pchilaka/1_Work/Packages/phase_diagram_workflows/notebooks/Aluminium_composition_scaling/Al_executor/avg.dat"
  return np.loadtxt(parts[0], usecols=usecols)
/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/cmmc/ptmp/pchilaka/mamba/envs/lammps_mdmc/lib/python3.12/site-packages/numpy/_core/_methods.py:180: RuntimeWarning: invalid val

RuntimeError: Calphy execution failed with IndexError: index -1 is out of bounds for axis 0 with size 0

In [10]:
# Show executor results
print("Calculation object:")
display(calphy_results_executor[0])
print("\nResults DataFrame:")
display(calphy_results_executor[1])
print("\nMetadata used:")
print(metadata_dict)

Calculation object:


NameError: name 'calphy_results_executor' is not defined

## 4. Comparison and Benefits

### Key Differences:
- **Traditional**: `execution_mode="library"`, no `lmp=` -- calphy builds and owns the
  `pylammpsmpi.LammpsLibrary` session itself
- **Executor-based**: same `execution_mode="library"`, but the session is built on an
  `executorlib` executor you manage and passed in via `lmp=`

### Benefits of Executor Integration:
- Reuse one managed LAMMPS session/resource allocation across multiple calls
- The same `SingleNodeExecutor` pattern extends directly to `SlurmClusterExecutor` for
  cluster-scale execution
- Metadata support for result caching and retrieval
- Full backward compatibility -- the traditional call path is unchanged

In [11]:
# Clean up executor
if 'executor' in locals():
    executor.shutdown()
    print("Executor shutdown complete")

print("Demo completed.")

Total wall time: 0:00:10


Executor shutdown complete
Demo completed.
